# Priors trained on every N-th frame: how does the reduction hurt with growing N?

The prior trained on every 10th frame was not worse than the authors' prior (see `evaluate_priors.ipynb`). This notebook trains the same prior on every **20th, 50th and 100th** frame and evaluates all of them on the same full test data.

Fixed for every N:
- **the split:** the official AMASS train / valid / test split of the given prior (`ORIGINAL_AMASS_SPLITS`);
- **the model and its training:** the setup of the pretrained prior `id_0900` (12 flow layers, lr 0.005, batch 2000), seed 10;
- **the number of iterations:** `MAX_ITERATIONS` = 2000, as for N = 10. So only the amount of data changes, and the training compute is the same. (With equal epochs instead, N = 100 would get 200 steps and be undertrained, which would mix two effects.)

Only the training data differ: every N-th frame of each sequence. The validation during training uses the compressed valid split, as for N = 10. The final comparison uses the **full** (not compressed) test split, Transitions + SSM.

Steps (each one skips what already exists, so the notebook can be stopped and run again):
1. **compress** the raw AMASS with `compress_amass` (every N-th frame) → `~/data/amass_compressed_c<N>`;
2. **rotation files** `<Dataset>.pt` with `preprocess_data.py` → `~/data/AMASS_rotations_c<N>`;
3. **train** with `scripts/train_prior.py` → `~/data/experiments/prior_c<N>/id_<9000 + N>`;
4. **evaluate** every prior on the full test split: the new ones, the existing N = 10 run and, for reference, the authors' prior.

Time on an M5 Pro CPU: compression and preprocessing ~15-30 min per N (reading the raw data), training ~1.8 h per N (2000 iterations x 3.3 s), evaluation a few minutes per prior. In total about 6-7 h; best overnight.

Note: there is no N = 1 on our local data (that needs the rotation files of the full training data, about 10x larger than for N = 10). The authors' prior is shown for orientation only: it was trained on a different AMASS release (most likely SMPL-X N; ours is SMPL-X G).

In [ ]:
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

from hp.core.evaluator import PriorEvaluator
from hp.core.prior_training import load_train_script, make_config
from hp.data import compression
from hp.data.amass_cfg import ORIGINAL_AMASS_SPLITS
from hp.utils.paths import REPO_ROOT, get_path

# ---- parameters ----------------------------------------------------------------------
STRIDES = [20, 30, 45, 66, 100]                          # every N-th frame, one prior each
COMPRESSED_RAW = "~/data/amass_compressed_c{n}"  # output of the compression for stride n
COMPRESSED_ROTATIONS = "~/data/AMASS_rotations_c{n}"
RUN_ROOT = "~/data/experiments/prior_c{n}"       # the run is saved in RUN_ROOT/id_<9000 + n>
EXISTING_RUNS = {10: "~/data/experiments/prior_c10/id_9001_0"}  # already trained, evaluated with the new ones
EXISTING_COMPRESSED = {10: "~/data/amass_compressed"}         # their compressed data (for the frame counts)
AUTHORS_PRIOR = REPO_ROOT / "experiments" / "pretrained_models" / "id_0900"  # setup of every run; evaluated for reference
FULL_ROTATIONS = "~/data/AMASS_rotations"        # rotation files of the FULL data: the final test runs on these
MAX_ITERATIONS = 2000            # the same for every N (as the N = 10 run); about 3.3 s per iteration on CPU
VALID_INTERVAL = 500
VALID_WASSERSTEIN = False        # the Wasserstein distance on the validation set is slow
DEVICE = "cpu"                   # "cpu", "mps" or "cuda:0" (on a Mac cpu is as fast as mps here)
SEED = 10
CLIP_GRAD_NORM = 1.0             # as the N = 10 run
SAVE_CKPT_EVERY = 1000
NUM_WORKERS = 4                  # processes for the compression
REPEATS = 3                      # log-likelihood evaluations per prior
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(str(p)).expanduser()
path = lambda pattern, n: expand(pattern.format(n=n))
DATASET_DIRS = {"EyesJapanDataset": "Eyes_Japan_Dataset"}  # folder names on disk that differ from the split names
needed = sorted({d for names in ORIGINAL_AMASS_SPLITS.values() for d in names})  # train + valid + test datasets
raw_dir = get_path("amass_raw")

assert all((expand(FULL_ROTATIONS) / f"{n}.pt").is_file() for n in ORIGINAL_AMASS_SPLITS["test"]), "the full test rotation files are missing"
assert expand(RUN_ROOT.format(n=0)).resolve() != (REPO_ROOT / "experiments").resolve(), "runs must not go into the tracked experiments folder"
print(f"raw AMASS: {raw_dir}")
print(f"strides {STRIDES}, {MAX_ITERATIONS} iterations each, datasets: {', '.join(needed)}")

## 1. Compress the raw AMASS

Every N-th frame of every sequence, no cropping, only the datasets of the split. Files that exist are kept (resume); an output folder made with other parameters is refused.

In [ ]:
for n in STRIDES:
    cfg = OmegaConf.create(dict(stride=n, crop_start_s=0.0, crop_end_s=0.0, output_dir=str(path(COMPRESSED_RAW, n)),
                                datasets=[DATASET_DIRS.get(d, d) for d in needed], raw_dir=raw_dir,
                                num_workers=NUM_WORKERS, overwrite=False))
    print(f"--- stride {n} -> {cfg.output_dir}")
    manifest = compression.run(cfg)
    t = manifest["total"]
    print(f"{t['sequences']:,} sequences, {t['frames_before']:,} -> {t['frames_after']:,} frames ({100 * t['frames_after'] / t['frames_before']:.1f}%)")

## 2. Rotation files



`<Dataset>.pt` of every dataset of the split, made from the compressed sequences (skipped if it exists).

In [ ]:
def split_frames(compressed_dir):
    """Frames per split of a compressed AMASS folder, from its manifest."""
    manifest = json.load(open(expand(compressed_dir) / compression.MANIFEST))
    return {split: sum(manifest["datasets"][DATASET_DIRS.get(d, d)]["frames_after"] for d in names)
            for split, names in ORIGINAL_AMASS_SPLITS.items()}


frames = {n: split_frames(d) for n, d in EXISTING_COMPRESSED.items()}  # stride -> split -> frames
for n in STRIDES:
    out = path(COMPRESSED_ROTATIONS, n)
    out.mkdir(parents=True, exist_ok=True)
    for name in needed:
        if not (out / f"{name}.pt").is_file():
            print(f"stride {n}: preprocessing {name} ...")
            subprocess.run([sys.executable, str(REPO_ROOT / "scripts" / "preprocess" / "preprocess_data.py"), "--dataset", name,
                            "--data_dir", str(path(COMPRESSED_RAW, n)), "--save_dir", str(out)], check=True, capture_output=True)
    frames[n] = split_frames(path(COMPRESSED_RAW, n))
    print(f"stride {n:3d}: " + ", ".join(f"{split} {f:,} frames" for split, f in frames[n].items()) + f"  -> {out}")

## 3. Train


One prior per N with `scripts/train_prior.py` (`main(config)`), on the compressed train split; validation on the compressed valid split. A finished run (a folder `id_<9000 + N>...` with `eval.json`) is not trained again. A new run never overwrites an old one (`prepare_saving_process` adds `_0`, `_1`, ...).

The numbers that `main` prints at the end (and stores in `eval.json`) are on the **compressed** test split; the comparison in step 4 uses the full one.

In [ ]:
def finished_run(root, exp_id):
    """The newest run folder of exp_id below root that finished (has eval.json), or None."""
    runs = [p for p in Path(root).glob(f"id_{exp_id}*") if (p / "eval.json").is_file() and (p / "model_ckpt_pdf_estimator.pt").is_file()]
    return max(runs, key=lambda p: p.stat().st_mtime) if runs else None


train_prior = load_train_script()
run_dirs = {n: expand(p) for n, p in EXISTING_RUNS.items()}
for n in STRIDES:
    exp_id, root = 9000 + n, path(RUN_ROOT, n)
    done = finished_run(root, exp_id)
    if done is not None:
        print(f"stride {n}: already trained in {done}")
        run_dirs[n] = done
        continue
    os.environ["HUPROSO_AMASS_ROTATIONS"] = str(path(COMPRESSED_ROTATIONS, n))  # this kernel only
    os.environ["HUPROSO_AMASS_SPLIT"] = "original"
    os.environ["HUPROSO_DEVICE"] = DEVICE
    os.environ.pop("HUPROSO_SUBSET", None)
    cfg = make_config(AUTHORS_PRIOR / "config.yaml", exp_id, str(root), MAX_ITERATIONS, VALID_INTERVAL, VALID_WASSERSTEIN,
                      DEVICE, SEED, CLIP_GRAD_NORM, SAVE_CKPT_EVERY)
    cfg.comment = f"prior trained on every {n}-th frame"
    print(f"stride {n}: {MAX_ITERATIONS:,} iterations x batch {cfg.batch_size} = "
          f"{MAX_ITERATIONS * cfg.batch_size / frames[n]['train']:.1f} epochs over {frames[n]['train']:,} training frames")
    train_prior.main(cfg)
    run_dirs[n] = Path(cfg.save.path_dir)  # main() put the real run folder here
    print(f"stride {n}: saved to {run_dirs[n]}")

## 4. Evaluate on the full test split
Every prior on the same data: the full (not compressed) official test split of our local AMASS, Transitions + SSM. Log-likelihood of `num_eval_samples` random test poses, repeated `REPEATS` times (higher is better), and the Wasserstein distance of generated poses to the test data (lower is better).

In [ ]:
os.environ["HUPROSO_AMASS_ROTATIONS"] = str(expand(FULL_ROTATIONS))  # from here on: the full test data
os.environ["HUPROSO_AMASS_SPLIT"] = "original"
os.environ.pop("HUPROSO_SUBSET", None)
n_full_test = sum(torch.load(expand(FULL_ROTATIONS) / f"{n}.pt").shape[0] for n in ORIGINAL_AMASS_SPLITS["test"])


def evaluate(model_dir):
    """Log-likelihood and Wasserstein distance of a prior on the full test split."""
    c = OmegaConf.load(Path(model_dir) / "config.yaml")
    c.only_eval, c.standard_mode, c.condition = True, True, False
    c.device = DEVICE
    c.save.path_dir = str(model_dir)
    np.random.seed(SEED), torch.manual_seed(SEED)
    trainer = getattr(importlib.import_module("hp.core.trainer"), c.trainer)(c)
    trainer.load_model(save_path=str(model_dir))
    n_test = len(trainer.eval_dataset)
    assert n_test == n_full_test, f"evaluated on {n_test} test frames, the full test set has {n_full_test}"
    ll = [trainer.eval()["eval_avg_ll"] for _ in range(REPEATS)]
    d = PriorEvaluator.eval(c, trainer, render_human=False, plot_samples=False, save_plot=False, compute_d_wasser=True)
    return dict(model_dir=str(model_dir), iterations=int(c.max_iterations), test_frames=n_test,
                ll=float(np.mean(ll)), ll_std=float(np.std(ll)), wasserstein=float(d["wasserstein_distance_ndim"]))


results = {"authors": evaluate(AUTHORS_PRIOR)}
print(f"authors' prior id_0900: log-likelihood {results['authors']['ll']:.2f}, Wasserstein {results['authors']['wasserstein']:.3f}")
for n in sorted(run_dirs):
    results[n] = evaluate(run_dirs[n])
    print(f"every {n:3d}-th frame:     log-likelihood {results[n]['ll']:.2f} +- {results[n]['ll_std']:.2f}, Wasserstein {results[n]['wasserstein']:.3f}")

## Result

In [ ]:
strides = sorted(n for n in results if n != "authors")
a = results["authors"]
print(f"full test split ({n_full_test:,} frames); higher log-likelihood and lower Wasserstein are better")
print(f"{'prior':28s} {'train frames':>12s} {'iterations':>10s} {'log-lik.':>9s} {'+-':>5s} {'Wasserstein':>11s}")
print(f"{'authors id_0900 (full data)':28s} {'-':>12s} {a['iterations']:10,} {a['ll']:9.2f} {a['ll_std']:5.2f} {a['wasserstein']:11.3f}")
for n in strides:
    r = results[n]
    train_frames = f"{frames[n]['train']:,}" if n in frames else "?"
    print(f"{f'every {n}-th frame':28s} {train_frames:>12s} {r['iterations']:10,} {r['ll']:9.2f} {r['ll_std']:5.2f} {r['wasserstein']:11.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, key, label in ((axes[0], "ll", "log-likelihood (higher is better)"), (axes[1], "wasserstein", "Wasserstein (lower is better)")):
    ax.plot(strides, [results[n][key] for n in strides], "o-", color="C0", label="ours (SMPL-X G, 2k it.)")
    if key == "ll":
        ax.errorbar(strides, [results[n]["ll"] for n in strides], yerr=[results[n]["ll_std"] for n in strides], fmt="none", color="C0")
    ax.axhline(a[key], color="0.5", ls="--", label="authors' id_0900 (other release, 20k it.)")
    ax.set_xscale("log")
    ax.set_xticks(strides, [str(n) for n in strides])
    ax.set_xlabel("every N-th frame")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
fig.suptitle("Priors on the full test split (Transitions + SSM)")
fig.tight_layout()
out = expand("~/data/experiments/stride_comparison")
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "stride_comparison.png", dpi=120)
plt.show()

with open(out / "stride_comparison.json", "w") as f:
    json.dump(dict(max_iterations=MAX_ITERATIONS, seed=SEED, frames={str(n): v for n, v in frames.items()},
                   results={str(k): v for k, v in results.items()}), f, indent=1)
print("saved to", out)

In [ ]:
# LL per iteration of every run: training (smoothed) and validation, with the epoch boundaries
SMOOTH = 50  # iterations of the moving average of the training LL

fig, (ax_tr, ax_va) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
lows, highs = [], []
for k, n in enumerate(sorted(run_dirs)):
    run = Path(run_dirs[n])
    if not (run / "eval.json").is_file():
        print(f"N = {n}: no eval.json yet in {run}, skipped")
        continue
    ev = json.load(open(run / "eval.json"))
    batch = OmegaConf.load(run / "config.yaml").batch_size
    ll = -np.asarray(ev["loss"], dtype=float)  # training LL of each batch
    it = np.arange(1, len(ll) + 1)
    smooth = np.convolve(ll, np.ones(SMOOTH) / SMOOTH, mode="valid")
    it_s = it[SMOOTH - 1:]
    color, label = f"C{k}", f"every {n}-th frame"
    if n in frames:
        per_epoch = frames[n]["train"] / batch  # iterations per epoch
        label += f" ({len(ll) / per_epoch:.1f} epochs)"
        ends = np.arange(per_epoch, len(ll) + 1, per_epoch)
        ax_tr.plot(ends, np.interp(ends, it_s, smooth), "|", color=color, ms=12, mew=1.5)
    ax_tr.plot(it, ll, color=color, alpha=0.12, lw=0.5)
    ax_tr.plot(it_s, smooth, color=color, lw=1.5, label=label)
    v = ev["valid_metrics"]
    ax_va.plot(v["iter"], -np.asarray(v["valid_loss"], dtype=float), "o-", color=color, label=label)
    lows.append(np.percentile(smooth, 5)), highs.append(smooth.max())

ax_tr.set_ylim(min(lows) - 5, max(highs) + 5)
ax_tr.set_ylabel("training LL per pose")
ax_tr.set_title(f"training LL ({SMOOTH}-iteration moving average, raw values faint); | = end of an epoch")
ax_va.set_ylabel("validation LL per pose")
ax_va.set_title("validation LL (each run on its own compressed valid split)")
ax_va.set_xlabel("iteration")
for ax in (ax_tr, ax_va):
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
out = expand("~/data/experiments/stride_comparison")
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "ll_per_iteration.png", dpi=120)
plt.show()